In [ ]:
# 本文件用于进行时间序列回归，通过将N个资产全部堆叠到一起回归得到暴露矩阵B，维度为K*N，K为因子个数，N为资产个数
%run config_0.ipynb
%run data_io_1.ipynb
%run clean_and_universe_2.ipynb
%run monthly_4.ipynb
%run portfolio_5.ipynb

import pandas as pd
import numpy as np
import statsmodels.api as sm

In [ ]:
# 提取被解释变量R，维度为T*N，同期时间，回归后体现共同变化（即暴露矩阵）
def get_R(monthly):
    R = monthly.pivot_table(
        index = ym , 
        columns = stock_code , 
        values = monthly_excess_ret
    )
    T = R.shape[0]
    N = R.shape[1]
    return R , T , N

In [ ]:
# 时间序列回归：对每只股票单独做 OLS（用各自的有效月份），得到 K*N 的暴露矩阵 B
def time_series_regression(R , F):
    F = sm.add_constant(F)
    factors = list(F.columns.drop('const'))

    B = pd.DataFrame(index = factors , columns = R.columns , dtype = float)
    alpha = pd.Series(index = R.columns , dtype = float)
    alpha_t = pd.Series(index = R.columns , dtype = float)   # 截距(定价误差)的 t 值

    for s in R.columns:
        y = R[s]
        mask = y.notna() & F.notna().all(axis = 1)
        # 有效样本数要大于参数个数（因子数 + 截距）
        if mask.sum() <= F.shape[1]:
            continue
        res = sm.OLS(y[mask] , F[mask]).fit(cov_type = 'HC1')   # White 异方差稳健标准误
        B[s] = res.params[factors]
        alpha[s] = res.params['const']
        alpha_t[s] = res.tvalues['const']

    return B , alpha , alpha_t

df = read_daily_data()
df = cleaning(df)
df = universe(df)
cf_df = read_cf_data()
cf_df = cf_cleaning(cf_df)
bs_df = read_bs_data()
bs_df = bs_cleaning(bs_df)
df = get_market_feature(df)
df = get_momentum(df)   # 与 monthly_4 / portfolio_5 的内联流程一致：动量列必须先构造
df = get_cp(df , cf_df)
df = get_asset_growth(df , bs_df)
df = get_excess_return(df)
df = get_bm(df)
df = get_ep(df)
df = get_one_month_volatility(df)
df = get_max_return(df)
df = get_illiquidity(df)
df = get_turnover_12m(df)
df = get_abn_turnover_1m(df)
monthly = get_ym_monthly_excessret(df)
snap = get_form_signal(df)
monthly_final = get_monthly_panel(monthly , snap)
F = get_groups_neutral(monthly_final)
R , T , N= get_R(monthly)
B , alpha , alpha_t = time_series_regression(R , F)
print('查看估计的暴露矩阵 B')
print(B)
print('查看定价误差 alpha 及其 t 值')
print(alpha)
print(alpha_t)